# Dot Plot of Scube1 Expression in Dorsal & Ventral Striatum

This notebook creates a dot plot of **Scube1** gene expression across cell subclasses
in the dorsal striatum (STRd) and ventral striatum (STRv) using 10x snRNA-seq data
from the Allen Brain Cell (ABC) Atlas.

Additional marker genes are included for cell-type identification:
- **Drd1** / **Drd2** — D1 vs D2 medium spiny neurons
- **Chat** — cholinergic interneurons
- **Pvalb** / **Sst** — GABAergic interneuron subtypes
- **Th** — dopaminergic/catecholaminergic marker
- **Foxp2** — expressed in a subset of MSNs
- **Aqp4** — astrocyte marker
- **Mog** — oligodendrocyte marker

Cortical cell types that may be incidentally captured in striatum dissections are excluded.

In [ ]:
import pandas as pd
import numpy as np
import re
import anndata
import scanpy as sc
import matplotlib.pyplot as plt
from pathlib import Path

from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache
from abc_atlas_access.abc_atlas_cache.anndata_utils import get_gene_data

## 1. Initialize the ABC Atlas Cache

In [ ]:
download_base = Path('../../data/abc_atlas')
abc_cache = AbcProjectCache.from_s3_cache(download_base)

print(f"Current manifest: {abc_cache.current_manifest}")

## 2. Load Cell Metadata and Taxonomy

In [ ]:
# Load cell metadata
cell = abc_cache.get_metadata_dataframe(
    directory='WMB-10X',
    file_name='cell_metadata',
    dtype={'cell_label': str}
)
cell.set_index('cell_label', inplace=True)
print(f"Total cells in WMB-10X: {len(cell):,}")

In [ ]:
# Load gene metadata
gene = abc_cache.get_metadata_dataframe(
    directory='WMB-10X',
    file_name='gene'
)
gene.set_index('gene_identifier', inplace=True)
print(f"Total genes: {len(gene):,}")

In [ ]:
# Load cluster taxonomy pivot table
cluster_details = abc_cache.get_metadata_dataframe(
    directory='WMB-taxonomy',
    file_name='cluster_to_cluster_annotation_membership_pivoted',
    keep_default_na=False
)
cluster_details.set_index('cluster_alias', inplace=True)

# Join taxonomy annotations onto cell metadata
cell_extended = cell.join(cluster_details, on='cluster_alias')
print(f"Taxonomy levels: {list(cluster_details.columns)}")

## 3. Filter to Striatum Cells and Exclude Cortical Types

In [ ]:
# Filter to dorsal and ventral striatum dissection regions
str_rois = ['STRd', 'STRv']
str_cells = cell_extended[cell_extended['region_of_interest_acronym'].isin(str_rois)].copy()
print(f"Cells in striatum regions: {len(str_cells):,}")
for roi, count in str_cells.groupby('region_of_interest_acronym').size().items():
    print(f"  {roi}: {count:,}")

# Show all subclasses present
print(f"\nAll subclasses found in striatum dissections:")
subclass_counts = str_cells.groupby('subclass', observed=True).size().sort_values(ascending=False)
for sc_name, count in subclass_counts.items():
    print(f"  {sc_name}: {count:,} cells")

In [ ]:
# Exclude cortical cell types that are incidentally captured in striatum dissections.
# These are identifiable by 'CTX' in their subclass name or by belonging to cortical classes.
cortex_patterns = ['CTX', 'L2/3', 'L4/5', 'L5 ', 'L6 ', 'L6b', 'RSP']

def is_cortical(subclass_name):
    """Return True if subclass name indicates a cortical cell type."""
    return any(pat in subclass_name for pat in cortex_patterns)

cortical_subclasses = [s for s in str_cells['subclass'].unique() if is_cortical(s)]
if cortical_subclasses:
    print(f"Excluding {len(cortical_subclasses)} cortical subclasses:")
    for s in sorted(cortical_subclasses):
        n = (str_cells['subclass'] == s).sum()
        print(f"  {s}: {n:,} cells")
    str_cells = str_cells[~str_cells['subclass'].isin(cortical_subclasses)].copy()
else:
    print("No cortical subclasses found — no exclusions needed.")

print(f"\nCells after excluding cortex types: {len(str_cells):,}")

In [ ]:
# Show final subclass breakdown
print("Striatum subclasses (after cortex exclusion):")
subclass_counts = str_cells.groupby('subclass', observed=True).size().sort_values(ascending=False)
for sc_name, count in subclass_counts.items():
    print(f"  {sc_name}: {count:,} cells")

## 4. Define Gene List

In [ ]:
# Primary gene of interest
primary_gene = ['Scube1']

# MSN identity markers
msn_markers = ['Drd1', 'Drd2', 'Foxp2']

# Interneuron / other neuron markers
interneuron_markers = ['Chat', 'Pvalb', 'Sst', 'Th']

# Glial markers
glial_markers = ['Aqp4', 'Mog']

all_genes_list = primary_gene + msn_markers + interneuron_markers + glial_markers

# Verify genes are present in the dataset
available_genes = gene[gene['gene_symbol'].isin(all_genes_list)]
found_symbols = set(available_genes['gene_symbol'])
missing = [g for g in all_genes_list if g not in found_symbols]
if missing:
    print(f"WARNING — genes not found in dataset: {missing}")

gene_list = [g for g in all_genes_list if g in found_symbols]
print(f"Genes for dot plot ({len(gene_list)}): {gene_list}")

## 5. Load Expression Data

In [ ]:
# Extract expression for selected genes from all striatum cells.
# This uses the anndata_utils helper which handles chunked reading
# across multiple h5ad files.
expression_data = get_gene_data(
    abc_atlas_cache=abc_cache,
    all_cells=str_cells,
    all_genes=gene,
    selected_genes=gene_list,
    data_type='log2'
)

# Drop any rows that are all NaN (cells not found in expression files)
expression_data = expression_data.dropna(how='all')
print(f"Expression data: {expression_data.shape[0]:,} cells x {expression_data.shape[1]} genes")

## 6. Build AnnData Object

In [ ]:
# Align cells: keep only cells present in both metadata and expression data
common_cells = str_cells.index.intersection(expression_data.index)
expression_data = expression_data.loc[common_cells, gene_list]
str_cells_aligned = str_cells.loc[common_cells]

# Add a region label for faceting
region_map = {'STRd': 'Dorsal striatum', 'STRv': 'Ventral striatum'}
str_cells_aligned['region_label'] = str_cells_aligned['region_of_interest_acronym'].map(region_map)

# Create AnnData
adata = anndata.AnnData(
    X=expression_data.values.astype(np.float32),
    obs=str_cells_aligned[['subclass', 'supertype', 'class', 'region_of_interest_acronym', 'region_label']].copy(),
    var=pd.DataFrame(index=gene_list)
)

# Create shorter display labels (strip leading number prefix)
adata.obs['subclass_short'] = adata.obs['subclass'].apply(
    lambda x: re.sub(r'^\d+\s+', '', x)
)

adata.obs['subclass_short'] = pd.Categorical(adata.obs['subclass_short'])
adata.obs['subclass'] = pd.Categorical(adata.obs['subclass'])

n_subclasses = adata.obs['subclass'].cat.categories.size
print(adata)
print(f"\n{n_subclasses} subclasses, {len(common_cells):,} cells")

## 7. Dot Plot: All Striatum Cell Subclasses

In [ ]:
# Define gene groups for the dot plot
gene_groups = {
    'Primary': [g for g in primary_gene if g in gene_list],
    'MSN markers': [g for g in msn_markers if g in gene_list],
    'Interneuron': [g for g in interneuron_markers if g in gene_list],
    'Glia': [g for g in glial_markers if g in gene_list],
}

outdir = Path('scube1')
outdir.mkdir(exist_ok=True)

dp = sc.pl.dotplot(
    adata,
    var_names=gene_groups,
    groupby='subclass_short',
    standard_scale='var',
    cmap='Reds',
    figsize=(12, max(6, n_subclasses * 0.5)),
    show=False,
    return_fig=True
)
dp.style(dot_edge_color='black', dot_edge_lw=0.5)
dp.savefig(outdir / 'dotplot_striatum_Scube1_by_subclass.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: scube1/dotplot_striatum_Scube1_by_subclass.png")

## 8. Split by Region: Dorsal vs Ventral Striatum

In [ ]:
for region in ['STRd', 'STRv']:
    adata_region = adata[adata.obs['region_of_interest_acronym'] == region].copy()
    region_name = region_map[region]
    n_sub = adata_region.obs['subclass_short'].nunique()

    # Skip subclasses with very few cells in this region
    min_cells = 10
    sub_counts = adata_region.obs.groupby('subclass_short', observed=True).size()
    valid_subs = sub_counts[sub_counts >= min_cells].index
    adata_region = adata_region[adata_region.obs['subclass_short'].isin(valid_subs)].copy()
    n_sub = len(valid_subs)

    print(f"\n{region_name}: {adata_region.n_obs:,} cells, {n_sub} subclasses")

    dp = sc.pl.dotplot(
        adata_region,
        var_names=gene_groups,
        groupby='subclass_short',
        standard_scale='var',
        cmap='Reds',
        figsize=(12, max(5, n_sub * 0.5)),
        title=f'Scube1 & markers — {region_name}',
        show=False,
        return_fig=True
    )
    dp.style(dot_edge_color='black', dot_edge_lw=0.5)
    fname = outdir / f'dotplot_{region}_Scube1_by_subclass.png'
    dp.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {fname}")

## 9. Summary Statistics

In [ ]:
# Mean expression and fraction expressing per subclass
expr_df = pd.DataFrame(
    adata.X,
    index=adata.obs.index,
    columns=adata.var.index
)
expr_df['subclass'] = adata.obs['subclass_short'].values
expr_df['region'] = adata.obs['region_of_interest_acronym'].values

print("=== Mean Scube1 expression by subclass and region ===")
summary = expr_df.groupby(['subclass', 'region']).agg(
    n_cells=('Scube1', 'size'),
    mean_expr=('Scube1', 'mean'),
    frac_expressing=('Scube1', lambda x: (x > 0).mean())
).round(4)
print(summary.to_string())